# Data Preprocessing
## Inventory Restocking RL Project
**Author:** Pratheesha 
**Purpose:** Clean the dataset and prepare it for the RL environment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# Make sure project root is in path
sys.path.append('..')
import config

print("Libraries imported successfully!")
print(f"Raw data path  : {config.DATA_RAW_PATH}")
print(f"Processed path : {config.DATA_PROCESSED_PATH}")

## Load Raw Data

In [ ]:
df = pd.read_csv(config.DATA_RAW_PATH)
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## Handle Missing Values

In [ ]:
print("Missing values BEFORE cleaning:")
print(df.isnull().sum())

# Fill numerical columns with median
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled '{col}' nulls with median: {median_val:.2f}")

# Fill categorical columns with mode
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled '{col}' nulls with mode: {mode_val}")

print("\nMissing values AFTER cleaning:")
print(df.isnull().sum())

## Remove Duplicates

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Removed {before - after} duplicate rows.")
print(f"Remaining rows: {after:,}")

## Parse Dates and Extract Time Features

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Day_of_Week'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

print("Date features added:")
print(df[['Date', 'Day_of_Week', 'Month', 'Year']].head(10))

## Select One Product for RL Environment

In [ ]:
# Select the product with the most complete records
product_counts = df['Product ID'].value_counts()
selected_product = product_counts.index[0]

print(f"Selected Product ID : {selected_product}")
print(f"Records for product : {product_counts[selected_product]:,}")

df_product = df[df['Product ID'] == selected_product].copy()
df_product = df_product.sort_values('Date').reset_index(drop=True)

print(f"\nDate range: {df_product['Date'].min()} → {df_product['Date'].max()}")
df_product.head()

## Create Promotion Flag

In [ ]:
# Create Promo_Flag from Discount column (1 if discount > 0, else 0)
# Adjust column name below if your dataset uses a different name
if 'Discount' in df_product.columns:
    df_product['Promo_Flag'] = df_product['Discount'].apply(
        lambda x: 1 if x > 0 else 0
    )
elif 'Promotion' in df_product.columns:
    df_product['Promo_Flag'] = df_product['Promotion'].astype(int)
else:
    # If no promotion column, default to 0 (no promotions)
    df_product['Promo_Flag'] = 0
    print("Warning: No promotion column found. Promo_Flag set to 0.")

print(f"Promo_Flag distribution:\n{df_product['Promo_Flag'].value_counts()}")

## Step 7: Discretise Stock Level into Bins

In [ ]:
# Discretise stock into 3 bins using config thresholds
# Low=0 (< 20 units), Medium=1 (20-59 units), High=2 (60+ units)

def discretise_stock(stock):
    if stock < config.STOCK_LOW_THRESHOLD:
        return 0   # Low
    elif stock < config.STOCK_HIGH_THRESHOLD:
        return 1   # Medium
    else:
        return 2   # High

df_product['Stock_Bin'] = df_product['Inventory Level'].apply(discretise_stock)

print("Stock bin distribution:")
bin_labels = {0: 'Low (0)', 1: 'Medium (1)', 2: 'High (2)'}
print(df_product['Stock_Bin'].map(bin_labels).value_counts())

# Visualise
plt.figure(figsize=(6, 4))
df_product['Stock_Bin'].value_counts().sort_index().plot(
    kind='bar', color=['red', 'orange', 'green'], edgecolor='black'
)
plt.title('Discretised Stock Level Distribution')
plt.xlabel('Stock Bin (0=Low, 1=Medium, 2=High)')
plt.ylabel('Days Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/plots/preprocessing_stock_bins.png', dpi=150)
plt.show()

## Step 8: Select Final Columns and Save

In [ ]:
# Select only the columns needed by the RL environment
final_columns = [
    'Date', 'Day_of_Week', 'Month', 'Year',
    'Units Sold', 'Inventory Level', 'Stock_Bin', 'Promo_Flag'
]

# Keep only columns that exist in the dataframe
final_columns = [col for col in final_columns if col in df_product.columns]
df_final = df_product[final_columns].copy()

print(f"Final dataset shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")
df_final.head()

In [ ]:
# Save preprocessed data
os.makedirs(os.path.dirname(config.DATA_PROCESSED_PATH), exist_ok=True)
df_final.to_csv(config.DATA_PROCESSED_PATH, index=False)
print(f"Saved to: {config.DATA_PROCESSED_PATH}")
print(f"File size: {os.path.getsize(config.DATA_PROCESSED_PATH) / 1024:.1f} KB")

## Preprocessing Complete 
The preprocessed file is saved to `data/processed/preprocessed_product.csv`.  
Other Members can now use this file via `config.DATA_PROCESSED_PATH`.